# dYdX collector catalog -> pandas

Instrument definitions come through Nautilus's own catalog API (`catalog.instruments()`), which decodes them into `CryptoPerpetual` objects; `to_dict()` is a staticmethod on each `Instrument` class, so it's called as `CryptoPerpetual.to_dict(i)`, not `i.to_dict()`.

Market-data rows come through the kernel's column-projected readers (`kernel.catalog_files`), the one read path research uses besides `BacktestDataConfig` streaming: they read only the columns a cell needs from the 1 s `DydxSecondSnapshot` files, without decoding every row's 20-level book. Reads below are always time-bounded (`start_ns`/`end_ns`) rather than pulling a full unbounded history into memory (MEM-01), since this catalog grows continuously as the collector keeps running.

In [ ]:
import sys

import pandas as pd

from nautilus_trader.model.instruments import CryptoPerpetual
from nautilus_trader.persistence.catalog import ParquetDataCatalog


# platform/, relative to this notebook: makes `kernel` importable.
if "../.." not in sys.path:
    sys.path.insert(0, "../..")

CATALOG_PATH = "../../data/catalog"  # relative to this notebook's directory
catalog = ParquetDataCatalog(CATALOG_PATH)
instrument_ids = [i.id.value for i in catalog.instruments()]
instrument_ids

In [ ]:
instruments_df = pd.DataFrame([CryptoPerpetual.to_dict(i) for i in catalog.instruments()])
instruments_df

In [ ]:
import time

from kernel.catalog_files import query_second_ohlc


# Bound the read -- an unbounded read materializes every second ever collected for this
# instrument into memory (MEM-01).
# Note: per-instrument retention pruning (archive.prune_catalog) may make the actual
# coverage shorter than LOOKBACK_HOURS if this instrument has a tighter retention.
LOOKBACK_HOURS = 24
end_ns = time.time_ns()
start_ns = end_ns - LOOKBACK_HOURS * 3_600 * 1_000_000_000

instrument_id = instrument_ids[0]
seconds = query_second_ohlc(CATALOG_PATH, instrument_id, start_ns, end_ns)

# A second with no trade has no close (its OHLC is None): there is no last price to plot for it.
closes = [(s.ts_event, s.close_price) for s in seconds if s.close_price is not None]
if not closes:
    print(f"No traded seconds for {instrument_id} in the last {LOOKBACK_HOURS}h.")
close_df = pd.DataFrame(closes, columns=["ts_event", "close"])
if not close_df.empty:
    close_df["ts_event"] = pd.to_datetime(close_df["ts_event"], unit="ns")
    close_df = close_df.set_index("ts_event").sort_index()
close_df

In [ ]:
if not close_df.empty:
    close_df["close"].plot(title=instrument_id)

## Reusing an existing indicator (never redefine inline)

Indicators are implemented once in `kernel.indicators` and imported here unmodified -- the same classes the live read models and the backtest strategies use (FR-10). A new indicator always gets added to that module, never redefined inline in a notebook cell.

`Microprice` is fed from each 1 s snapshot's top of book (level 0 of its bid/ask arrays), read by `kernel.catalog_files.query_top_of_book` -- this catalog does not store `QuoteTick` data (see `platform/CLAUDE.md`'s 1s-snapshot signal architecture), and microprice is derived on read, never stored (SIGNAL-01). A second with an empty book side has no top of book and is not returned.

In [ ]:
from kernel.catalog_files import query_top_of_book
from kernel.indicators import Microprice


# Same window and instrument as the close series above.
tops = query_top_of_book(CATALOG_PATH, instrument_id, start_ns, end_ns)

microprice = Microprice()
micro_series = []
for top in tops:
    # Microprice is a pure function of one top of book, and update_raw is a no-op on a zero
    # total size: resetting first makes `initialized` mean "this second produced a value", so a
    # skipped second is never plotted with the previous second's value.
    microprice.reset()
    microprice.update_raw(top.bid_price, top.bid_size, top.ask_price, top.ask_size)
    if microprice.initialized:
        micro_series.append((top.ts_event, microprice.value))

if not micro_series:
    print(f"No top-of-book snapshots for {instrument_id} in the last {LOOKBACK_HOURS}h.")
else:
    micro_df = pd.DataFrame(micro_series, columns=["ts_event", "microprice"])
    micro_df["ts_event"] = pd.to_datetime(micro_df["ts_event"], unit="ns")
    micro_df.set_index("ts_event").sort_index()["microprice"].plot(
        title=f"{instrument_id} microprice"
    )